# 05 - Module 2: Stability

Evaluates Lipschitz, Kendall tau, Coefficient of Variation, and Identity
across the same configurations. Raw Lipschitz and CoV values can span
many orders of magnitude, so both a log-scaled and a min-max normalized
version are reported.


In [ ]:
import numpy as np
import pandas as pd

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.metrics.stability import evaluate_stability
from xai_blockchain_framework.models import load_ml_model
from xai_blockchain_framework.utils import save_csv
from xai_blockchain_framework.utils.normalization import log_normalize, min_max_normalize
from xai_blockchain_framework.xai import LimeTabularExplainer, ShapTreeExplainer

set_seed()
ell = np.load(PATHS.data_processed / 'elliptic_splits.npz')
eth = np.load(PATHS.data_processed / 'ethereum_splits.npz')
ell_idx = np.load(PATHS.results_dir / 'xai_eval_indices_elliptic.npy')
eth_idx = np.load(PATHS.results_dir / 'xai_eval_indices_ethereum.npy')


## Compute stability for each configuration

In [ ]:
rows = []
for model_name in ['randomforest', 'lightgbm']:
    for ds, splits, indices in [('Elliptic', ell, ell_idx), ('Ethereum', eth, eth_idx)]:
        model = load_ml_model(PATHS.models_dir / f'{ds.lower()}_{model_name}.joblib')
        for xai in ['shap', 'lime']:
            attrs = np.load(PATHS.results_dir / f'{xai}_{model_name}_{ds.lower()}.npy')
            if xai == 'shap':
                explainer = ShapTreeExplainer(model)
                explain_fn = explainer.make_explain_fn()
            else:
                explainer = LimeTabularExplainer(splits['X_train'], random_state=CONFIG.random_seed)
                explain_fn = explainer.make_explain_fn(model.predict_proba)
            metrics = evaluate_stability(
                splits['X_test'], attrs, indices, explain_fn,
                rng=np.random.default_rng(CONFIG.random_seed),
            )
            rows.append({
                'Type': 'ML', 'Model': model_name.upper(),
                'XAI': xai.upper(), 'Dataset': ds, **metrics,
            })
df = pd.DataFrame(rows)
print(df.to_string(index=False))


## Add log and min-max normalized columns

In [ ]:
df['Lipschitz_log']  = np.log10(1 + df['Lipschitz'])
df['Lipschitz_norm'] = log_normalize(df['Lipschitz'], lower_better=True)
df['CoV_log']        = np.log10(1 + df['CoV_Bootstrap'])
df['CoV_norm']       = log_normalize(df['CoV_Bootstrap'], lower_better=True)
df['Kendall_norm']   = min_max_normalize(df['Kendall_tau'], lower_better=False)
df['Identity_norm']  = min_max_normalize(df['Identity'], lower_better=False)
df['Stability_Score'] = (
    df['Lipschitz_norm'] + df['Kendall_norm'] + df['CoV_norm'] + df['Identity_norm']
) / 4
df = df.sort_values('Stability_Score', ascending=False)
save_csv(df, PATHS.results_dir / 'module2_stability_results.csv')
